# Santorini V3 Supervised Bootstrap on Kaggle

Use this notebook with a Kaggle GPU accelerator to bootstrap the V3 Santorini architecture from the known-good V2 replay buffer.

Expected input Dataset contents:

- `latest.examples`: the `training6_v2` replay buffer

The local source artifact is currently:

- `temp/santorini_kaggle_training6_v2/latest.examples`

The bootstrap creates a fresh 8-block/96-channel V3 model, trains with a deterministic validation split, saves the best checkpoint by validation loss, and writes metadata for later evaluation.

## 0. P100 PyTorch Compatibility Guard

Run this before importing `torch`. Kaggle's preinstalled CUDA wheel may not support Tesla P100 (`sm_60`), so this cell switches P100 sessions to the official CUDA 12.6 wheel.

In [ ]:
import subprocess
import sys

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip()
except Exception:
    gpu_name = ""

print("GPU:", gpu_name or "not detected")

if "P100" in gpu_name:
    print("Detected P100; installing CUDA 12.6 PyTorch wheel...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "torch",
        "torchvision",
        "torchaudio",
    ])
else:
    print("No P100-specific install needed.")

## 1. Check Runtime

In [ ]:
import os
import torch

print("/kaggle/working exists:", os.path.exists("/kaggle/working"))
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Get the Repo

Enable Internet in the Kaggle notebook settings before running the clone and pip cells. If Internet is disabled, attach the repo as a Dataset instead and set `REPO_DIR` to that copied or mounted path.

Set `REPO_BRANCH` to the branch that contains `bootstrap_santorini_v3.py` and V3 architecture support.

In [ ]:
REPO_URL = "https://github.com/Luminous9/alpha-zero-general.git"
REPO_DIR = "/kaggle/working/alpha-zero-general"
REPO_BRANCH = "main"

In [ ]:
%cd /kaggle/working
!test -d "$REPO_DIR/.git" && git -C "$REPO_DIR" fetch --all || git clone "$REPO_URL" "$REPO_DIR"
!git -C "$REPO_DIR" checkout "$REPO_BRANCH"
!git -C "$REPO_DIR" pull --ff-only || true
%cd $REPO_DIR
!git rev-parse --short HEAD
!test -f bootstrap_santorini_v3.py && echo "Found bootstrap_santorini_v3.py"

## 3. Install Dependencies

In [ ]:
!pip install coloredlogs tqdm

## 4. Configure Bootstrap Inputs

Attach the Kaggle Dataset that contains the `training6_v2/latest.examples` file, then set `BOOTSTRAP_SOURCE` to that read-only `/kaggle/input/...` folder. The notebook creates a symlink in `/kaggle/working` so the replay file does not need to be copied.

In [ ]:
import os
import shutil

# Example: "/kaggle/input/santorini-kaggle-training6-v2"
BOOTSTRAP_SOURCE = "/kaggle/input/santorini-kaggle-training6-v2"
BOOTSTRAP_EXAMPLES = "latest.examples"
BOOTSTRAP_WORK = "/kaggle/working/Santorini-AZ/v3_bootstrap_input"
BOOTSTRAP_OUTPUT = "/kaggle/working/Santorini-AZ/v3_bootstrap"

os.makedirs(BOOTSTRAP_WORK, exist_ok=True)
os.makedirs(BOOTSTRAP_OUTPUT, exist_ok=True)

def link_or_copy(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    try:
        os.symlink(src, dst)
        print("Symlinked", dst, "->", src)
    except OSError:
        shutil.copy2(src, dst)
        print("Copied", src, "->", dst)

examples_src = os.path.join(BOOTSTRAP_SOURCE, BOOTSTRAP_EXAMPLES)
examples_dst = os.path.join(BOOTSTRAP_WORK, BOOTSTRAP_EXAMPLES)
if not os.path.isfile(examples_src):
    raise FileNotFoundError(examples_src)
link_or_copy(examples_src, examples_dst)

print("Examples:", examples_dst)
!ls -lh "$BOOTSTRAP_WORK"

## 5. Inspect Replay Buffer

In [ ]:
import pickle

examples_path = os.path.join(BOOTSTRAP_WORK, BOOTSTRAP_EXAMPLES)
with open(examples_path, "rb") as f:
    history = pickle.Unpickler(f).load()

history_lengths = [len(window) for window in history]
print("history windows:", len(history_lengths))
print("total examples:", sum(history_lengths))
print("history lengths:", history_lengths)
sample_board, sample_policy, sample_value = history[0][0]
print("sample board shape:", getattr(sample_board, "shape", None))
print("sample policy shape:", getattr(sample_policy, "shape", None))
print("sample policy sum:", float(sum(sample_policy)))
print("sample value:", sample_value)
del history

## 6. Supervised V3 Bootstrap

This starts from fresh V3 random weights and trains against the selected replay buffer. The script saves `best.pth.tar`, `final.pth.tar`, and `bootstrap_metadata.json` in `BOOTSTRAP_OUTPUT`.

In [ ]:
BOOTSTRAP_EPOCHS = 15
BOOTSTRAP_BATCH_SIZE = 512
BOOTSTRAP_PATIENCE = 3
BOOTSTRAP_SEED = 7

!python bootstrap_santorini_v3.py \
  --examples-file "$examples_path" \
  --output-folder "$BOOTSTRAP_OUTPUT" \
  --architecture v3 \
  --epochs "$BOOTSTRAP_EPOCHS" \
  --batch-size "$BOOTSTRAP_BATCH_SIZE" \
  --validation-fraction 0.10 \
  --patience "$BOOTSTRAP_PATIENCE" \
  --seed "$BOOTSTRAP_SEED" \
  --quiet

## 7. Sanity Check Bootstrapped Checkpoint

In [ ]:
import json
import numpy as np

from santorini.SantoriniGame import SantoriniGame
from santorini.pytorch.NNet import build_nnet

metadata_path = os.path.join(BOOTSTRAP_OUTPUT, "bootstrap_metadata.json")
with open(metadata_path) as f:
    metadata = json.load(f)

print("best epoch:", metadata.get("best_epoch"))
print("best validation loss:", metadata.get("best_validation_loss"))
print("completed epochs:", metadata.get("completed_epochs"))

game = SantoriniGame(5, true_random_placement=True)
nnet = build_nnet(game, "v3")
nnet.load_checkpoint(BOOTSTRAP_OUTPUT, "best.pth.tar")

board = game.getCanonicalForm(game.getInitBoard(), 1)
policy, value = nnet.predict(board)
print("policy shape:", policy.shape)
print("policy sum:", float(np.sum(policy)))
print("value:", float(value))

## 8. List Output Artifacts

In [ ]:
!find "$BOOTSTRAP_OUTPUT" -maxdepth 1 -type f -printf "%f %k KB\n" | sort
!du -sh "$BOOTSTRAP_OUTPUT"

## 9. Optional: Prepare A Normal Checkpoint Folder

Use this when you want the next Kaggle cell or notebook to treat the bootstrapped V3 model like a standard Santorini checkpoint folder. Do not copy `latest.examples` here unless you intentionally want the normal training loop to load this replay buffer.

In [ ]:
import shutil

CHECKPOINT = "/kaggle/working/Santorini-AZ/v3_checkpoints"
os.makedirs(CHECKPOINT, exist_ok=True)
shutil.copy2(os.path.join(BOOTSTRAP_OUTPUT, "best.pth.tar"), os.path.join(CHECKPOINT, "best.pth.tar"))
shutil.copy2(os.path.join(BOOTSTRAP_OUTPUT, "bootstrap_metadata.json"), os.path.join(CHECKPOINT, "bootstrap_metadata.json"))
print("Prepared checkpoint folder:", CHECKPOINT)
!ls -lh "$CHECKPOINT"

## 10. Optional: Self-Play Smoke Test

This is deliberately tiny. It only checks that a V3 checkpoint can enter the normal AlphaZero loop.

In [ ]:
!python main_santorini.py \
  --preset local \
  --architecture v3 \
  --load-model \
  --load-folder "$CHECKPOINT" \
  --checkpoint "$CHECKPOINT" \
  --num-iters 1 \
  --num-eps 2 \
  --num-mcts-sims 4 \
  --arena-compare 2 \
  --epochs 1 \
  --batch-size 64 \
  --self-play-batch-size 2 \
  --arena-batch-size 2 \
  --quiet